# 07 — GPU Runtime & Dual-Coder Server Benchmarking (`Agent 07`)

- **Active Run ID**: `run_20261111_002`
- **Owner**: Agent 07 (`src/runtime/`, `src/model_runtime.py`)
- **Canonical Output**: `runs/run_20261111_002/gpu_runtime_config.json` (`gpu_runtime_config_v1`)

## Mission & Architectural Invariants
1. **Runtime Hardware Detection**: Detect physical accelerator and available VRAM at runtime (never assume hardware from documentation).
2. **Model-Family Diversity**:
   - **Coder A**: DeepSeek reasoning-oriented 32B quantized model (`deepseek-ai/DeepSeek-R1-Distill-Qwen-32B-AWQ`).
   - **Coder B**: Independent multilingual model family (`mistralai/Mistral-Small-3.1-24B-Instruct-2503-AWQ` preferred; `google/gemma-3-27b-it-qat-q4_0-gguf` alternative). Qwen-derived substitutes for Coder B are strictly blocked.
3. **5-Stage Runtime Benchmark & Empirically Derived Memory Budgets**:
   - Stage 1: Coder A alone
   - Stage 2: Coder B alone
   - Stage 3: Both servers resident
   - Stage 4: One simultaneous structured request to each
   - Stage 5: Bounded concurrent requests & failure recovery
4. **Crash-Safe Atomic Question Checkpoints & Sequential Fallback**:
   - Every accepted response records `run_id`, `question_id`, `coder_id`, `prompt_hash`, `source_hash`, `model_repo`, `model_revision`, `quantization`, `generation_config`, `raw_response`, `parsed_response`, `schema_status`, `runtime_sec`, and `peak_vram_mb`.


In [4]:
import os
import subprocess
import sys
from pathlib import Path

def _find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/HFG'), Path('/content')]
    for p in candidates:
        if (p / 'src').is_dir() and ((p / 'pyproject.toml').is_file() or (p / 'data').is_dir()):
            return p.resolve()
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is None and Path('/content').is_dir():
    target = Path('/content/HFG')
    print(f'Repository not found locally; cloning into {target}...')
    try:
        subprocess.run(['git', 'clone', 'https://github.com/necantis/HFG.git', str(target)], check=True)
        REPO_ROOT = _find_repo_root()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Automatic git clone failed. If the repo is private, clone it manually or mount Google Drive and cd into HFG.'
        ) from exc

if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate HFG repo root from {Path.cwd()}. Clone/open inside HFG first.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'HFG repo root: {REPO_ROOT}')

import json
from src.runtime import (
    detect_hardware_profile,
    build_and_save_runtime_config,
    validate_model_family_diversity,
    DEFAULT_CODER_A_SPEC,
    DEFAULT_CODER_B_MISTRAL_SPEC,
    ALTERNATIVE_CODER_B_GEMMA_SPEC,
)

hw = detect_hardware_profile()
print(json.dumps(hw.to_dict(), indent=2))


HFG repo root: /content/HFG
{
  "measured_at_utc": "2026-11-13T18:39:28.259584+00:00",
  "hostname": "7d4213ec7641",
  "os_platform": "Linux 6.6.159+ (x86_64)",
  "cpu_model": "AMD EPYC 9B45",
  "cpu_logical_cores": 48,
  "system_ram_total_mb": 181130.01,
  "system_ram_available_mb": 178099.12,
  "gpu_available": true,
  "gpu_count": 1,
  "gpu_device_name": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "gpu_total_vram_mb": 97887.0,
  "gpu_free_vram_mb": 97251.0,
  "gpu_used_vram_mb": 0.0,
  "cuda_version": "13.0",
  "driver_version": "580.82.07",
  "compute_capability": "12.0",
  "detection_method": "nvidia-smi"
}


In [5]:
# Validate model-family diversity (Coder A DeepSeek 32B vs Coder B Mistral Small 3.1 24B / Gemma 3 27B)
validate_model_family_diversity(DEFAULT_CODER_A_SPEC, DEFAULT_CODER_B_MISTRAL_SPEC)
validate_model_family_diversity(DEFAULT_CODER_A_SPEC, ALTERNATIVE_CODER_B_GEMMA_SPEC)
print('Model-family diversity verified successfully.')


Model-family diversity verified successfully.


In [6]:
# Execute the 5-stage runtime benchmark and derive memory budgets from measurements
cfg, bench_report, cfg_path = build_and_save_runtime_config(run_id='run_20261111_002')
print(f'Saved config to: {cfg_path}')
print(json.dumps(bench_report['derived_memory_budget'], indent=2))


Saved config to: /content/HFG/runs/run_20261111_002/gpu_runtime_config.json
{
  "execution_mode": "CONCURRENT_DUAL_SERVER",
  "concurrent_feasible": true,
  "single_gpu_feasible": true,
  "gpu_memory_fraction": 0.5449,
  "coder_a_memory_fraction": 0.2945,
  "coder_b_memory_fraction": 0.2504,
  "reserved_vram_headroom_mb": 44548.37,
  "reserved_vram_headroom_fraction": 0.4551,
  "coder_a_total_required_mb": 26692.0,
  "coder_b_total_required_mb": 22692.0,
  "combined_required_mb": 49384.0,
  "rationale": "Concurrent dual-coder resident serving fits within 97251 MB free VRAM (combined requirement 49384 MB, headroom 11746 MB)."
}
